In [ ]:
# !pip install obonet
# !pip install biopython
# !pip install transformers torch torchvision torchaudio

In [ ]:
import obonet, networkx, torch
import pandas as pd, numpy as np
from Bio import SeqIO
from sklearn.preprocessing import MultiLabelBinarizer
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

In [ ]:
# Load training annotations (protein GO term associations)
train_terms_path = "/kaggle/input/cafa-6-protein-function-prediction/Train/train_terms.tsv"
train_terms_df = pd.read_csv(train_terms_path, sep="\t", header=0)
print(f"initial train dataframe shape: {train_terms_df.shape}")

# Load Train Taxonomy file
train_taxon_path = "/kaggle/input/cafa-6-protein-function-prediction/Train/train_taxonomy.tsv"
train_taxon_df = pd.read_csv(train_taxon_path, sep="\t", header=None)
train_taxon_df.columns = ['ProtID', 'Taxon']
train_taxon_map = train_taxon_df.set_index("ProtID").to_dict()['Taxon']
train_terms_df['Taxon'] = train_terms_df['EntryID'].map(train_taxon_map).map(str)

# Select the most prevalent taxonomies that comprise 95% of records
train_taxon_cdf = train_terms_df['Taxon'].value_counts().cumsum() / train_terms_df.shape[0]
selected_train_taxon = train_taxon_cdf[train_taxon_cdf <= 0.95].index.tolist()

train_terms_df = train_terms_df\
                .loc[train_terms_df['Taxon'].isin(selected_train_taxon)]\
                .reset_index(drop=True)

print(f"filtered train dataframe shape: {train_terms_df.shape}")

In [ ]:
# ######### run this cell to see how well the test dataset is represented in the filtered train dataset
# test_fasta_path = "/kaggle/input/cafa-6-protein-function-prediction/Test/testsuperset.fasta"
# test_fasta_sequences = SeqIO.parse(open(test_fasta_path),'fasta')
# from collections import defaultdict
# test_taxon_counts = defaultdict(int)
# for fasta in test_fasta_sequences:
#     taxon = fasta.description.split(" ")[-1]
#     test_taxon_counts[taxon] +=1

# print(f"unique test taxon count: {len(test_taxon_counts.keys())}")
# train_taxon_list = train_terms_df['Taxon'].unique().tolist()
# common_taxon = set(test_taxon_counts.keys()) & (set(train_taxon_list))
# print(f"common train and test taxon count: {len(common_taxon)}")
# print(f"total test sequences: {sum(test_taxon_counts.values())}")
# test_common_taxon_seq_count = sum([v for (k,v) in test_taxon_counts.items() if k in train_taxon_list])
# print(f"test sequences with taxon present in train dataset: {test_common_taxon_seq_count}")

# # TODO: create predictions for only these taxonomies from the test superset.
#     # We can also measure the performance for other taxonomies using a train/validation split.
#     # Or, we can measure the performance on the records removed from the train set. (the remaining 1 to 5 percent)

In [ ]:
# Load Ontology File
ont_file_path = "/kaggle/input/cafa-6-protein-function-prediction/Train/go-basic.obo"
ont_graph = obonet.read_obo(ont_file_path) # ignore_obsolete: bool = True
reversed_ont_graph = ont_graph.reverse(copy=False)

In [ ]:
def propagate_terms(term_set):
    expanded = term_set.copy()
    stack = list(term_set)
    while stack:
        cur = stack.pop()
        for p in list(reversed_ont_graph.predecessors(cur)):
            if p not in expanded:
                expanded.add(p)
                stack.append(p)
    return expanded

def prune_terms(term_set):
    pruned = term_set.copy()
    for t in list(terms):
        descent_stack = []
        descent_stack.extend(list(ont_graph.predecessors(t)))
        while descent_stack:
            child = descent_stack.pop()
            if child in terms:
                pruned.discard(t)
                break
            else:
                descent_stack.extend(list(ont_graph.predecessors(child)))
    return pruned

term_ns_map = {k: v['namespace'] for k,v in ont_graph.nodes(data=True)}
aspect_ns_map = {
    'biological_process': "P",
    'cellular_component': "C",
    'molecular_function': "F"
}

In [ ]:
propagated_rows = []
pruned_rows = []
for prot, group in train_terms_df.groupby("EntryID"):
    terms = set(group['term'])
    # Create Propagated Records
    prop_terms = propagate_terms(terms)
    # Create Pruned Records
    pruned_terms = prune_terms(terms)
    
    propagated_rows.append({"EntryID": prot, "term": list(prop_terms), "Taxon": group['Taxon'].iloc[0]})
    pruned_rows.append({"EntryID": prot, "term": list(pruned_terms), "Taxon": group['Taxon'].iloc[0]})

train_terms_df_propagated = pd.DataFrame(propagated_rows).explode("term")
train_terms_df_propagated['aspect'] = train_terms_df_propagated['term']\
    .map(term_ns_map)\
    .map(aspect_ns_map)

train_terms_df_pruned = pd.DataFrame(pruned_rows).explode("term")
train_terms_df_pruned['aspect'] = train_terms_df_pruned['term']\
    .map(term_ns_map)\
    .map(aspect_ns_map)

In [ ]:
pruned_goterm_count_df = train_terms_df_pruned.groupby(["aspect", "term"], as_index = False)\
    .agg(pruned_count = ("EntryID", "count"))

prop_goterm_count_df = train_terms_df_propagated.groupby(["aspect", "term"], as_index = False)\
    .agg(prop_count = ("EntryID", "count"))

goterm_count_df = prop_goterm_count_df.merge(
    pruned_goterm_count_df,
    how = 'left',
    on = ["aspect", "term"]
)
goterm_count_df.fillna({'pruned_count': 0}, inplace=True)
goterm_count_df['specificity'] = goterm_count_df['pruned_count'] / goterm_count_df['prop_count']

In [ ]:
# For each sub-ontology/namespace/aspect, filter GO terms (classes for classification/ranking task) based on:
    # Applying a minimum sample size (MSS) or class size
    # Applying a minimum specificity level (MSL)

MSL = 0.2

MSS_BP = goterm_count_df.loc[goterm_count_df['aspect'] == "P", "pruned_count"].quantile(0.8)
MSS_CC = goterm_count_df.loc[goterm_count_df['aspect'] == "C", "pruned_count"].quantile(0.8)
MSS_MF = goterm_count_df.loc[goterm_count_df['aspect'] == "F", "pruned_count"].quantile(0.8)

selected_go_terms = goterm_count_df.loc[
    (
        ((goterm_count_df['pruned_count'] >= MSS_BP) & (goterm_count_df['aspect'] == "P"))
        |((goterm_count_df['pruned_count'] >= MSS_CC) & (goterm_count_df['aspect'] == "C"))
        |((goterm_count_df['pruned_count'] >= MSS_MF) & (goterm_count_df['aspect'] == "F"))
    )
    &(goterm_count_df['specificity'] >= MSL), "term"
]

train_df_final = train_terms_df_pruned.loc[train_terms_df_pruned['term'].isin(selected_go_terms)]
print(f"final GO terms count: {len(selected_go_terms)}")
print(f"coverage of pruned records: {train_df_final.shape[0] / train_terms_df_pruned.shape[0]}")
print(train_df_final.shape)
display(train_df_final.head(3))

In [ ]:
# Assume train_df_final is already available as a DataFrame with columns ['EntryID', 'GO_term']
# 1. Group GO terms by each protein EntryID
grouped = train_df_final.groupby('EntryID')['term'].apply(list).reset_index(name='go_terms_list')

# 2. Binarize the GO term lists into a multi-label binary matrix
mlb = MultiLabelBinarizer()
label_matrix = mlb.fit_transform(grouped['go_terms_list'])

# Convert to a DataFrame for convenience (columns are GO term IDs, rows are proteins)
label_df = pd.DataFrame(label_matrix, index=grouped['EntryID'], columns=mlb.classes_)

# `label_df` now has one row per protein, with 1/0 indicating presence/absence of each GO term.
print(label_df.shape)
# print("Example protein labels (first row):\n", label_df.iloc[0])

In [ ]:
# 1. Load training sequences from FASTA
train_fasta_path = "/kaggle/input/cafa-6-protein-function-prediction/Train/train_sequences.fasta"
standrad_aa_string = "ACDEFGHIKLMNPQRSTVWY"

sequence_dict = {}
for record in SeqIO.parse(train_fasta_path, "fasta"):
    seq_id = record.id.split("|")[1]  # assuming this matches EntryID
    seq_str = str(record.seq)
    sequence_dict[seq_id] = seq_str

In [ ]:
# import matplotlib.pyplot as plt
# import numpy as np
# model_max_seq_len = 1022
# seq_len_list = list(map(len, sequence_dict.values()))
# plt.hist(seq_len_list, bins = 50);
# cvr = (np.array(seq_len_list) <= model_max_seq_len).mean().round(2) * 100
# plt.axvline(model_max_seq_len, label = f"model max sequence length covering {cvr}% records", color = "Red")
# plt.legend()

In [ ]:
# 2. Preprocess sequences: uppercase and replace invalid chars, truncate if needed
def preprocess_sequence(seq):
    seq = seq.strip().upper()
    # Replace any character not in 20 standard amino acids (ACDEFGHIKLMNPQRSTVWY) with 'X'
    seq = ''.join([ch if ch in standrad_aa_string else 'X' for ch in seq])
    # Truncate to max length (1022 for ESM, leaving room for <cls>)
    max_len = 1022
    if len(seq) > max_len:
        seq = seq[:max_len]
    return seq

grouped['sequence'] = grouped['EntryID'].map(sequence_dict).map(preprocess_sequence)

In [ ]:
# 3. Load the ESM-2 model and tokenizer
model_name = "facebook/esm2_t33_650M_UR50D"
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    do_lower_case=False # do_lower_case=False to preserve uppercase
)
model = AutoModel.from_pretrained(model_name)
# model = torch.nn.DataParallel(model) # if using T4x2
model.eval()
model.to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))

# 4. Embed sequences in batches
train_embeddings = []
batch_size = 8
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


for i in tqdm(
    range(0, len(grouped), batch_size),
    total = len(grouped)//batch_size + 1,
    desc="Creating Train Embeddings:"
):
    batch_seqs = list(grouped['sequence'][i:i+batch_size])
    # Tokenize the batch of sequences.
    # Note: For ESM, each amino acid is a token; the tokenizer will add the CLS token.
    encodings = tokenizer(batch_seqs, return_tensors='pt', padding=True, truncation=True, max_length = 1024)
    input_ids = encodings['input_ids'].to(device)
    attention_mask = encodings['attention_mask'].to(device)
    # =====if using T4x2
    # input_ids = encodings['input_ids'].to("cuda:0")
    # attention_mask = encodings['attention_mask'].to("cuda:0")
    # ==================
    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
        # Get the embeddings (last hidden state of [CLS] token for each sequence)
        last_hidden = outputs.last_hidden_state  # shape: (batch_size, seq_len, hidden_dim)
        # The [CLS] token embedding is at index 0 for each sequence
        cls_embeddings = last_hidden[:, 0, :].detach().cpu()
    train_embeddings.append(cls_embeddings)

train_embeddings = torch.cat(train_embeddings, dim=0).numpy()  # shape: (num_train_proteins, hidden_dim)
print("Training embeddings shape:", train_embeddings.shape)

In [ ]:
# 1. Load and preprocess test sequences
test_seqs = []
test_ids = []
for record in SeqIO.parse("testsuperset.fasta", "fasta"):
    test_ids.append(record.id)
    test_seqs.append(preprocess_sequence(str(record.seq)))

# 2. Embed test sequences using the same model and tokenizer
test_embeddings = []
batch_size = 16
for i in range(0, len(test_seqs), batch_size):
    batch = test_seqs[i:i+batch_size]
    encodings = tokenizer(batch, return_tensors='pt', padding=True, truncation=True)
    input_ids = encodings['input_ids'].to(device)
    attention_mask = encodings['attention_mask'].to(device)
    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
        cls_embeds = outputs.last_hidden_state[:, 0, :].detach().cpu()
    test_embeddings.append(cls_embeds)
test_embeddings = torch.cat(test_embeddings, dim=0).numpy()  # shape: (num_test_proteins, embed_dim)

In [ ]:
# 3. Normalize embeddings for cosine similarity
# Normalize training embeddings (to unit vectors)
train_norms = np.linalg.norm(train_embeddings, axis=1, keepdims=True)
train_emb_normed = train_embeddings / (train_norms + 1e-8)
# Normalize test embeddings
test_norms = np.linalg.norm(test_embeddings, axis=1, keepdims=True)
test_emb_normed = test_embeddings / (test_norms + 1e-8)

In [ ]:
# 4. Perform KNN search for each test protein
k = 30
predictions = {}

prediction_number_limit = 1500
total_go_terms = len(mlb.classes_)
selection_percentile = np.round(1 - prediction_number_limit / total_go_terms, 2)

for idx, test_vec in enumerate(test_emb_normed):
    # Compute cosine similarity between this test vector and all train vectors
    # dot product since both are unit norm gives cosine similarity
    sims = np.dot(train_emb_normed, test_vec) 
    # Get indices of top-k most similar training proteins
    top_k_idx = np.argpartition(-sims, k)[:k]
    top_k_sims = sims[top_k_idx]

    # Retrieve the binary label vectors of these top-k neighbors
    neighbor_label_vectors = label_matrix[top_k_idx]  # shape: (k, num_GO_terms)

    # Compute weighted avg of neighbor label vectors, weighted by similarity
    weighted_avg = np.dot(top_k_sims, neighbor_label_vectors) / k  # shape: (num_GO_terms,)

    threshold = np.quantile(weighted_avg, selection_percentile)
    predicted_terms = [(GO, score) for GO, score in zip(mlb.classes_, weighted_avg) if score >= threshold]

    predictions[test_ids[idx]] = predicted_terms

In [ ]:
first_test_id = test_ids[0]
print(f"Test protein: {first_test_id}")
print("Predicted GO terms:", predictions[first_test_id])